In [2]:
import pandas as pd
import numpy as np
import os
import glob
import re

# ================= 路径配置 =================
DATA_PATH = 'F:/self_quant/data/data/'
KLINE_DIR = os.path.join(DATA_PATH, '日K线')
PROFIT_DIR = os.path.join(DATA_PATH, '利润')
BALANCE_DIR = os.path.join(DATA_PATH, '资产负债')
CASHFLOW_DIR = os.path.join(DATA_PATH, '现金流量')
OUTPUT_DIR = os.path.join(DATA_PATH, '未退市合并后数据')

# 如果输出文件夹不存在，则创建
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ================= 列名配置 =================
income_columns = [
    '报告日', '净利润', '营业总收入', '营业总成本', '营业成本',
    '营业利润', '利润总额', '所得税费用', '归属于母公司所有者的净利润'
]

balance_columns = [
    '报告日', '货币资金', '应收账款', '存货', '流动资产合计',
    '非流动资产合计', '资产总计', '流动负债合计', '非流动负债合计',
    '负债合计', '所有者权益(或股东权益)合计', '归属于母公司所有者权益(或股东权益)合计'
]

cashflow_columns = [
    '报告日', '经营活动产生的现金流量净额', '投资活动产生的现金流量净额',
    '筹资活动产生的现金流量净额', '现金及现金等价物净增加额'
]

# ================= 核心功能函数 =================

def load_financial_statement(file_path, columns):
    """读取单个财务表并清洗"""
    if not file_path or not os.path.exists(file_path):
        return pd.DataFrame(columns=columns)

    try:
        df_full = pd.read_csv(file_path, encoding='gbk')
    except Exception as e:
        print(f"读取报错 {file_path}: {e}")
        return pd.DataFrame(columns=columns)

    valid_columns = [col for col in columns if col in df_full.columns]
    df = df_full[valid_columns].copy()

    if '报告日' in df.columns:
        # 将格式类似 20241231 转换为日期
        df['报告日'] = pd.to_datetime(df['报告日'], format='%Y%m%d', errors='coerce')
        # 去除没有报告日期的空行
        df = df.dropna(subset=['报告日'])

    return df

def find_file_by_code(directory, code_6digit):
    """根据6位股票代码在指定文件夹中模糊查找文件"""
    search_pattern = os.path.join(directory, f"*{code_6digit}*.csv")
    files = glob.glob(search_pattern)
    return files[0] if len(files) > 0 else None

def process_single_stock(kline_file):
    """处理并合并单只股票的所有数据"""
    # 1. 提取6位数字股票代码 (例如从 'sh.600003_ST东北高_日K.csv' 提取 '600003')
    filename = os.path.basename(kline_file)
    match = re.search(r'\d{6}', filename)
    if not match:
        return None

    code_6digit = match.group()

    # 2. 读取K线数据
    try:
        kline_df = pd.read_csv(kline_file, encoding='gbk')
    except:
        kline_df = pd.read_csv(kline_file, encoding='utf-8')

    if 'date' not in kline_df.columns:
        return None

    # K线日期格式化并排序（必须排序才能使用 merge_asof）
    kline_df['date'] = pd.to_datetime(kline_df['date'])
    kline_df = kline_df.sort_values('date')

    # 3. 寻找对应的财务报表文件路径
    profit_file = find_file_by_code(PROFIT_DIR, code_6digit)
    balance_file = find_file_by_code(BALANCE_DIR, code_6digit)
    cashflow_file = find_file_by_code(CASHFLOW_DIR, code_6digit)

    # 4. 加载三大表
    profit_df = load_financial_statement(profit_file, income_columns)
    balance_df = load_financial_statement(balance_file, balance_columns)
    cashflow_df = load_financial_statement(cashflow_file, cashflow_columns)

    # 5. 合并三大财报（根据'报告日' outer join）
    fin_df = pd.DataFrame(columns=['报告日'])
    if not profit_df.empty:
        fin_df = pd.merge(fin_df, profit_df, on='报告日', how='outer')
    if not balance_df.empty:
        fin_df = pd.merge(fin_df, balance_df, on='报告日', how='outer')
    if not cashflow_df.empty:
        fin_df = pd.merge(fin_df, cashflow_df, on='报告日', how='outer')

    # 如果三大表全为空，直接返回原始K线（或跳过）
    if fin_df.empty or len(fin_df.columns) == 1:
        return kline_df

    # ================= 核心逻辑：避免未来函数 =================
    # 假设报告日期的披露存在滞后性，例如 2024-12-31 的数据到 2025-03-31 才披露
    # 我们将财报的生效日向后推3个月（使用 pd.DateOffset）
    fin_df['实际披露日'] = fin_df['报告日'] + pd.DateOffset(months=3)

    # 清理并排序（必须按照合并键排序）
    fin_df = fin_df.dropna(subset=['实际披露日'])
    fin_df = fin_df.sort_values('实际披露日')

    # ================= 拼接到日K线 =================
    # 使用 merge_asof: 针对K线的每一个 'date'，向后寻找距离它最近且 '实际披露日' <= 'date' 的财务数据
    # direction='backward' 意味着使用“历史最新可用数据”
    merged_df = pd.merge_asof(
        kline_df,
        fin_df,
        left_on='date',
        right_on='实际披露日',
        direction='backward'
    )

    return merged_df

# ================= 主程序：批量处理 =================
if __name__ == '__main__':
    # 获取所有的K线文件
    kline_files = glob.glob(os.path.join(KLINE_DIR, "*.csv"))
    print(f"共找到 {len(kline_files)} 个K线文件，开始处理...")

    for kline_file in kline_files:
        stock_name = os.path.basename(kline_file).replace('_日K.csv', '')
        print(f"正在处理: {stock_name}")

        merged_df = process_single_stock(kline_file)

        if merged_df is not None and not merged_df.empty:
            output_path = os.path.join(OUTPUT_DIR, f"{stock_name}_合并数据.csv")
            # 保存为csv，忽略索引
            merged_df.to_csv(output_path, index=False, encoding='gbk')

    print("全部合并完成！数据保存在:", OUTPUT_DIR)

共找到 5192 个K线文件，开始处理...
正在处理: sh.600000_浦发银行
正在处理: sh.600004_白云机场
正在处理: sh.600006_东风股份
正在处理: sh.600007_中国国贸
正在处理: sh.600008_首创环保
正在处理: sh.600009_上海机场
正在处理: sh.600010_包钢股份
正在处理: sh.600011_华能国际
正在处理: sh.600012_皖通高速
正在处理: sh.600015_华夏银行
正在处理: sh.600016_民生银行
正在处理: sh.600017_日照港
正在处理: sh.600018_上港集团
正在处理: sh.600019_宝钢股份
正在处理: sh.600020_中原高速
正在处理: sh.600021_上海电力
正在处理: sh.600022_山东钢铁
正在处理: sh.600023_浙能电力
正在处理: sh.600025_华能水电
正在处理: sh.600026_中远海能
正在处理: sh.600027_华电国际
正在处理: sh.600028_中国石化
正在处理: sh.600029_南方航空
正在处理: sh.600030_中信证券
正在处理: sh.600031_三一重工
正在处理: sh.600032_浙江新能
正在处理: sh.600033_福建高速
正在处理: sh.600035_楚天高速
正在处理: sh.600036_招商银行
正在处理: sh.600037_歌华有线
正在处理: sh.600038_中直股份
正在处理: sh.600039_四川路桥
正在处理: sh.600048_保利发展
正在处理: sh.600050_中国联通
正在处理: sh.600051_宁波联合
正在处理: sh.600052_东望时代
正在处理: sh.600053_九鼎投资
正在处理: sh.600054_黄山旅游
正在处理: sh.600055_万东医疗
正在处理: sh.600056_中国医药
正在处理: sh.600057_厦门象屿
正在处理: sh.600058_五矿发展
正在处理: sh.600059_古越龙山
正在处理: sh.600060_海信视像
正在处理: sh.600061_国投资本
正在处理: sh.600062_华润双鹤
正在处理: sh.600